In [ ]:
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
    import math

    # =========================
    # 0) CARREGAR O CSV (FRED)
    # =========================
    # Se o arquivo estiver na mesma pasta do .py, deixe assim:
    caminho_csv = "DEXBZUS.csv"
    # Se você estiver rodando no sandbox daqui, seria:
    # caminho_csv = "/mnt/data/DEXBZUS.csv"

    df = pd.read_csv(caminho_csv)

    # Remove linhas vazias (às vezes vem "." no FRED)
    df = df[df["DEXBZUS"] != "."].copy()
    df["DEXBZUS"] = df["DEXBZUS"].astype(float)

    # =========================
    # 1) ESCOLHA DO SINAL
    # =========================
    # Para mercado financeiro, geralmente é melhor usar RETORNOS (não o preço bruto).
    USAR_RETORNO_LOG = True

    precos = df["DEXBZUS"].to_numpy(dtype=float)

    if USAR_RETORNO_LOG:
        # retornos log: r_t = ln(P_t) - ln(P_{t-1})
        lista_base = np.diff(np.log(precos)).tolist()
    else:
        # preço bruto
        lista_base = precos.tolist()

    print("Tamanho da série usada =", len(lista_base))

    # =========================
    # 2) SUAS FUNÇÕES
    # =========================

    # desvios acumulados (chamei de y)
    def calcular_Y(lista):
        media = sum(lista) / len(lista)
        y = []
        soma_aux = 0
        for x in lista:
            soma_aux += (x - media)
            y.append(soma_aux)
        return y

    # (Range)
    def calcular_R(y):
        amplitude = max(y) - min(y)
        return amplitude

    # desvio padrão
    def calcular_S(lista):
        n = len(lista)
        media = sum(lista) / n

        soma_quadrados = sum([(x - media)**2 for x in lista])
        variancia = soma_quadrados / n
        desvio_padrao = variancia**0.5

        return desvio_padrao

    def calcular_RS(lista_original, lista_y):
        R = max(lista_y) - min(lista_y)

        n = len(lista_original)
        media = sum(lista_original) / n
        soma_quadrados = sum((x - media)**2 for x in lista_original)
        S = (soma_quadrados / n)**0.5

        return R / S


    # =========================
    # 3) POTÊNCIAS DE 2 ATÉ O FINAL
    # =========================
    n_valores = []
    n = 2
    while n <= len(lista_base):
        n_valores.append(n)
        n *= 2

    print("n_valores =", n_valores)

    # =========================
    # 4) CALCULAR R/S PARA CADA PREFIXO
    # =========================
    lista_final_RS = []

    for n in n_valores:
        lista_n = lista_base[:n]
        y_n = calcular_Y(lista_n)
        rs_n = calcular_RS(lista_n, y_n)
        lista_final_RS.append(rs_n)

    # =========================
    # 5) GRÁFICO ANTES DO LOG (SÓ PONTOS)
    # =========================
    def plotar_rs_pontos(n_valores, rs_valores):
        plt.figure(figsize=(10, 6))
        plt.scatter(n_valores, rs_valores, label="Pontos (n, R/S)")
        plt.title("Análise R/S - Escala Linear (Somente Pontos)")
        plt.xlabel("Tamanho da Amostra (n)")
        plt.ylabel("Razão R/S")
        plt.grid(True, which="both", ls="-", alpha=0.3)
        plt.legend()
        plt.show()

    plotar_rs_pontos(n_valores, lista_final_RS)

    # =========================
    # 6) LOG-LOG BASE 2 + AJUSTE
    # =========================
    def plotar_rs_loglog(n_valores, rs_valores, base="2"):
        n = np.array(n_valores, dtype=float)
        rs = np.array(rs_valores, dtype=float)

        mask = rs > 0
        n = n[mask]
        rs = rs[mask]

        if base == "2":
            x = np.log2(n)
            y = np.log2(rs)
            xlabel = "log2(n)"
            ylabel = "log2(R/S)"
        elif base == "10":
            x = np.log10(n)
            y = np.log10(rs)
            xlabel = "log10(n)"
            ylabel = "log10(R/S)"
        else:
            x = np.log(n)
            y = np.log(rs)
            xlabel = "ln(n)"
            ylabel = "ln(R/S)"

        plt.figure(figsize=(10, 6))
        plt.scatter(x, y, color="blue", label="Dados em log-log")
        plt.title("Análise R/S - Escala Log-Log")
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.grid(True, which="both", ls="-", alpha=0.3)
        plt.legend()
        plt.show()

    def ajustar_reta_loglog_e_plotar(n_valores, rs_valores, base="2"):
        n = np.array(n_valores, dtype=float)
        rs = np.array(rs_valores, dtype=float)

        mask = rs > 0
        n = n[mask]
        rs = rs[mask]

        if base == "2":
            x = np.log2(n)
            y = np.log2(rs)
            xlabel = "log2(n)"
            ylabel = "log2(R/S)"
        elif base == "10":
            x = np.log10(n)
            y = np.log10(rs)
            xlabel = "log10(n)"
            ylabel = "log10(R/S)"
        else:
            x = np.log(n)
            y = np.log(rs)
            xlabel = "ln(n)"
            ylabel = "ln(R/S)"

        # Ajuste linear: y ≈ a + b*x
        b, a = np.polyfit(x, y, 1)  # slope=b, intercept=a
        y_hat = a + b * x

        plt.figure(figsize=(10, 6))
        plt.scatter(x, y, color="blue", label="Dados em log-log")
        plt.plot(x, y_hat, color="red", linestyle="--", label="Reta ajustada")
        plt.title("Ajuste Linear em Log-Log (R/S vs n)")
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.grid(True, which="both", ls="-", alpha=0.3)
        plt.legend()
        plt.show()

        return a, b

    # log2-log2 + ajuste
    plotar_rs_loglog(n_valores, lista_final_RS, base="2")
    a, b = ajustar_reta_loglog_e_plotar(n_valores, lista_final_RS, base="2")

    print("Parâmetros do ajuste em log2-log2:")
    print("Intercepto a =", a)
    print("Inclinação  b =", b)
    print("Expoente (H) =", b)
    print("Usou retorno log?" , USAR_RETORNO_LOG)

ModuleNotFoundError: No module named 'pandas'